# Prompt Engineering

**Learning goal:** Design a system prompt that reliably guides Claude to extract pharmaceutical regulatory fields — with few-shot examples, negative prompting, and a chain-of-thought instruction.

**Concepts covered**
- System message structure and persona setting
- Few-shot examples embedded in the system prompt
- Negative prompting — telling the model what NOT to do
- Chain-of-thought instruction
- Testing prompt quality by running the same input with and without examples

## Cell 1 — Setup

In [1]:
from dotenv import load_dotenv
import os
import anthropic

# Load ANTHROPIC_API_KEY from the project-root .env (one folder up from week1/).
load_dotenv(dotenv_path=os.path.join('..', '.env'))

assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY not found. Add it to the project-root .env file.'
)

# The client reads ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

MODEL = 'claude-haiku-4-5'   # supports temperature / top_k / top_p (Opus 4.8 / 4.7 do not)
print(f'Client ready. Using model: {MODEL}')

Client ready. Using model: claude-haiku-4-5


## Cell 2 — System Prompt (No Examples — Baseline)

**What this cell does**

This is the **baseline** experiment: we ask Claude to extract structured fields using *only* a plain instruction-based system prompt — **no examples, no rules**. Later cells add few-shot examples and negative prompting so we can measure the difference.

The cell has three parts:

**1. `SYSTEM_PROMPT_BASELINE`** — the system prompt (the model's standing instructions). It:
- Sets a **persona**: *"You are SmartIntake, a regulatory affairs intake specialist."*
- Defines the **five fields** to extract (`query_type`, `regulation_ref`, `product_area`, `urgency`, `submitting_team`), each constrained to a fixed list of allowed values (an *enum*).
- Asks for the answer as plain text. Notice it gives **no worked examples and no rules** — that's deliberate, so we can see how the model behaves on instructions alone.

**2. `ask(system, user_message)`** — a helper that calls the Claude API once and prints the result:
- `client.messages.create(...)` sends the request. `system=` carries the standing instructions; `messages=[{"role": "user", ...}]` carries the actual query. `max_tokens=300` caps the reply length.
- `response.content[0].text` is the model's text answer — `content` is a list of blocks, and `[0]` grabs the first one.
- `response.usage` reports **token counts** (`input_tokens` / `output_tokens`). These matter because few-shot examples in later cells make the prompt longer and therefore more expensive — we print them to compare cost.

**3. `TEST_QUERY` + `ask(...)`** — a sample regulatory message (a serious SUSAR that must be reported to the EMA within 15 days). The trailing `\` lets the string continue on the next line. The final `ask(SYSTEM_PROMPT_BASELINE, TEST_QUERY)` runs the baseline so you can eyeball how well a bare prompt does before we improve it.

> **Why a baseline?** You can only prove that few-shot examples and rules help if you first measure the version *without* them. This

In [2]:
SYSTEM_PROMPT_BASELINE = """
You are SmartIntake, a regulatory affairs intake specialist.
Extract the following fields from the user's regulatory query:
- query_type: one of complaint, submission, variation, safety_signal,
              label_update, inspection, general_enquiry
- regulation_ref: one of FDA_21CFR, EMA_CTR, ICH_E2A, ICH_Q10,
                  CDSCO_NDC, GxP_GMP, GxP_GCP, other
- product_area: one of oncology, cardiovascular, infectious_disease,
                cmc, clinical, labelling, general
- urgency: one of routine, standard, urgent, critical
- submitting_team: the team or function submitting the query

Respond in plain text listing each field and its value.
"""

def ask(system, user_message):
    response = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=system,
        messages=[{"role": "user", "content": user_message}]
    )
    print(response.content[0].text)
    print(f"\nTokens — in: {response.usage.input_tokens}, out: {response.usage.output_tokens}")

TEST_QUERY = "PV team here. We have a serious unexpected SUSAR for study NCT-2244 \
and need to report to EMA within 15 days per ICH E2A."

ask(SYSTEM_PROMPT_BASELINE, TEST_QUERY)

**INTAKE EXTRACTION**

- **query_type**: safety_signal
- **regulation_ref**: ICH_E2A
- **product_area**: clinical
- **urgency**: urgent
- **submitting_team**: PV (Pharmacovigilance)

Tokens — in: 233, out: 65


## Cell 3 — System Prompt (With Few-Shot Examples)

In [3]:
SYSTEM_PROMPT_FEW_SHOT = """
You are SmartIntake, a regulatory affairs intake specialist.
Extract the following fields from the user's regulatory query:
- query_type: one of complaint, submission, variation, safety_signal,
              label_update, inspection, general_enquiry
- regulation_ref: one of FDA_21CFR, EMA_CTR, ICH_E2A, ICH_Q10,
                  CDSCO_NDC, GxP_GMP, GxP_GCP, other
- product_area: one of oncology, cardiovascular, infectious_disease,
                cmc, clinical, labelling, general
- urgency: one of routine, standard, urgent, critical
- submitting_team: the team or function submitting the query

RULES:
- Never infer urgency from tone alone. Ask if deadline is unclear.
- submitting_team must be a team or function, never an individual name.
- Use 'other' for regulation_ref if the framework is not clearly identifiable.
- Think step by step about the regulatory context before choosing values.

EXAMPLE 1:
User: "We need to respond to an FDA query on our CMC section for NDA-209114."
Output:
query_type: submission
regulation_ref: FDA_21CFR
product_area: cmc
urgency: [ask — no deadline stated]
submitting_team: [ask — not provided]

EXAMPLE 2:
User: "PV team here. We have a new serious unexpected SUSAR for the Phase III trial."
Output:
query_type: safety_signal
regulation_ref: ICH_E2A
product_area: clinical
urgency: critical
submitting_team: PV
"""

ask(SYSTEM_PROMPT_FEW_SHOT, TEST_QUERY)

```
query_type: safety_signal
regulation_ref: ICH_E2A
product_area: clinical
urgency: urgent
submitting_team: PV
```

**Rationale:**
- **query_type: safety_signal** — SUSAR (Serious Unexpected Suspected Adverse Reaction) is a safety signal requiring expedited reporting
- **regulation_ref: ICH_E2A** — Explicitly stated; this is the clinical safety reporting standard
- **product_area: clinical** — Safety data from an active study (Phase III)
- **urgency: urgent** — 15-day reporting deadline to EMA creates time-bound compliance requirement (not routine, but 15 days provides a defined window for preparation)
- **submitting_team: PV** — Pharmacovigilance team clearly identified

**Note:** Confirm whether the 15-day clock has already started (from event onset, IND awareness, or report receipt) to ensure your submission timeline is accurate.

Tokens — in: 443, out: 229


## Cell 4 — Negative Prompting Test

**What this cell does**

This cell is an **A/B test** of *negative prompting* — it feeds the **same** query to both system prompts and prints the two answers side by side so you can compare them.

**The trick is in the test query:**

```
"Hi, can someone please urgently review our labelling update? No rush really."
```

It deliberately sends **contradictory urgency signals** — the word *"urgently"* pulls one way, *"No rush really"* pulls the other, and there's no real deadline. It's a trap designed to expose how each prompt handles ambiguity.

**Why run it twice — without vs. with negative prompting:**

| Run | Prompt used | What it lacks / has |
|-----|-------------|---------------------|
| **Without** | `SYSTEM_PROMPT_BASELINE` | No rules. The model is free to read the tone and will likely latch onto *"urgently"* and label `urgency: urgent` — even though the user said there's no rush. |
| **With** | `SYSTEM_PROMPT_FEW_SHOT` | Contains the **negative-prompting rule**: *"Never infer urgency from tone alone. Ask if deadline is unclear."* The model should **refuse to guess** and instead respond with something like `urgency: [ask — no deadline stated]`. |

**What "negative prompting" means here:** telling the model what **NOT** to do (*don't infer urgency from tone*) rather than only what to do. Running both prompts on the identical query isolates the rule as the only variable, so any difference in the `urgency` field is caused by the negative prompt — that's what proves the guardrail works.

> **Why this matters in pharma:** wrongly marking a routine request as `urgent` (or missing a genuine emergency) has real operational and compliance consequences. Negative prompting makes the model **conservative and predictable** instead of reacting to emotional wording — exactly what a regulated intake system needs.


In [5]:
# Test the urgency rule: does the model ask rather than infer?
casual_query = "Hi, can someone please urgently review our labelling update? No rush really."
print("=== Without negative prompting ===")
ask(SYSTEM_PROMPT_BASELINE, casual_query)

print("\n=== With negative prompting ===")
ask(SYSTEM_PROMPT_FEW_SHOT, casual_query)

=== Without negative prompting ===
Here is my extraction of the fields from your query:

**query_type:** label_update
**regulation_ref:** other
**product_area:** labelling
**urgency:** routine
**submitting_team:** unknown

---

> **⚠️ Note on Urgency Assessment:**
> Your message contains conflicting urgency signals — you initially said **"urgently"** but then clarified **"No rush really."** I have defaulted to **routine** based on your clarification, but please confirm the correct urgency level, as this affects processing priority. If this is genuinely time-sensitive, please resubmit with:
> - A specific deadline or regulatory timeline
> - The reason for urgency (e.g., pending submission, safety issue)

> **📋 Additional Information Needed:**
> To complete intake, please also provide:
> - Your **team or function name** (submitting_team was not identified)
> - The **product area/therapeutic area** involved
> - Any **relevant regulation or guidance** driving the label update

Tokens — in:

## Cell 5 — Observations

In [ ]:
# Write your observations here as a markdown cell:
# 1. Which fields improved with few-shot examples?
# 2. Did negative prompting change the urgency behaviour?
# 3. How many more tokens did the few-shot prompt use?
#    Is the quality improvement worth the cost?

### Key Takeaway

Few-shot examples and negative prompting cost tokens but substantially improve extraction reliability. For a regulated domain like pharma, the quality improvement is worth the cost — a missed urgency or wrong `query_type` has real consequences.